# Bike Demand Modeling
## Notebook 2: Modeling, Diagnostics, and Robustness Checks

In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.formula.api as smf

from src.preprocess import load_and_prepare
from src.models import (
    fit_count_models, fit_log_models,
    comparison_table_count, comparison_table_log,
    run_cv_count, run_cv_log
)
from src.diagnostics import (
    diagnostic_summary, diagnostic_plots,
    compute_vif, influence_table, cooks_plot,
    robust_se_table, bootstrap_se
)

In [ ]:
DATA_PATH = '../data/raw/day.csv'
day, day_cv = load_and_prepare(DATA_PATH)

## 1. Model Fitting

### 1.1 Count Response Models

| Model | Formula |
|---|---|
| Model 1 | `cnt ~ temp + hum + windspeed` |
| Model 2 | `cnt ~ temp + hum + windspeed + season + workingday + yr` |
| Model 3 | `cnt ~ temp + temp_sq + hum + windspeed + season + workingday + yr` |
| Model 4 | `cnt ~ temp * season + hum + windspeed + workingday + yr` |

In [ ]:
models = fit_count_models(day)
model1, model2, model3, model4 = (
    models["model1"], models["model2"], models["model3"], models["model4"]
)

### 1.2 Log-Response Models

In [ ]:
log_models = fit_log_models(day)
log_model1, log_model2, log_model3, log_model4 = (
    log_models["log_model1"], log_models["log_model2"],
    log_models["log_model3"], log_models["log_model4"]
)

### 1.3 Model Summaries

In [ ]:
print(model1.summary())

In [ ]:
print(model2.summary())

In [ ]:
print(model3.summary())

In [ ]:
print(model4.summary())

## 2. Model Comparison

### 2.1 In-sample fit (count models)

In [ ]:
comp_cnt = comparison_table_count(models)
comp_cnt.round(3)

### 2.2 In-sample fit (log models)

In [ ]:
comp_log = comparison_table_log(log_models)
comp_log.round(3)

### 2.3 Cross-validated RMSE (10-fold)

In [ ]:
cv_cnt = run_cv_count(day, day_cv)
cv_cnt.round(3)

In [ ]:
cv_log = run_cv_log(day, day_cv)
cv_log.round(3)

### 2.4 Final comparison tables

In [ ]:
final_comparison = comp_cnt.merge(cv_cnt, on="Model", how="left")
final_comparison.round(3)

In [ ]:
# Log models: merge on renamed key
cv_log_renamed = cv_log.rename(columns={"Model": "Model_log"})
final_log_comparison = comp_log.assign(Model_log=comp_log["Model"]).merge(
    cv_log_renamed, on="Model_log", how="left"
).drop(columns=["Model_log"])
final_log_comparison.round(3)

## 3. Statistical Diagnostic Summary

In [ ]:
diag_cnt = pd.concat([
    diagnostic_summary(model2, "Model 2"),
    diagnostic_summary(model3, "Model 3"),
    diagnostic_summary(model4, "Model 4"),
], ignore_index=True)
diag_cnt.round(4)

In [ ]:
diag_log = pd.concat([
    diagnostic_summary(log_model2, "Log Model 2"),
    diagnostic_summary(log_model3, "Log Model 3"),
    diagnostic_summary(log_model4, "Log Model 4"),
], ignore_index=True)
diag_log.round(4)

## 4. Diagnostic Plots (Residual Analysis)

In [ ]:
diagnostic_plots(model2, "Model 2", x_for_resid=day["temp"], x_label="Temperature")

In [ ]:
diagnostic_plots(model3, "Model 3", x_for_resid=day["temp"], x_label="Temperature")

In [ ]:
diagnostic_plots(model4, "Model 4", x_for_resid=day["temp"], x_label="Temperature")

In [ ]:
diagnostic_plots(log_model4, "Log Model 4", x_for_resid=day["temp"], x_label="Temperature")

## 5. Multicollinearity Check (VIF)

In [ ]:
vif_table = compute_vif(day)
vif_table.round(3)

## 6. Influence Diagnostics

In [ ]:
n_obs = len(day)
infl_compare = pd.concat([
    influence_table(model2, "Model 2", n_obs),
    influence_table(model3, "Model 3", n_obs),
    influence_table(model4, "Model 4", n_obs),
], ignore_index=True)
infl_compare

### Cook's Distance Plots

In [ ]:
cooks_plot(model2, "Model 2")

In [ ]:
cooks_plot(model3, "Model 3")

In [ ]:
cooks_plot(model4, "Model 4")

## 7. Robustness Checks

### 7.1 HC3 Robust Standard Errors (count Model 4)

In [ ]:
robust_cnt = robust_se_table(model4, "Model 4")
robust_cnt.round(4)

### 7.2 HC3 Robust Standard Errors (log Model 4)

In [ ]:
robust_log = robust_se_table(log_model4, "Log Model 4")
robust_log.round(4)

### 7.3 Bootstrap Standard Errors (count Model 4, B=1000)

In [ ]:
boot_table = bootstrap_se(
    day,
    formula="cnt ~ temp * C(season) + hum + windspeed + C(workingday) + C(yr)",
    B=1000,
    seed=42
)

# merge with HC3
model4_hc3 = model4.get_robustcov_results(cov_type="HC3")
boot_table["SE_classical"] = model4.bse.values
boot_table["SE_HC3"] = model4_hc3.bse
boot_table[["term", "coef", "SE_classical", "SE_HC3", "SE_bootstrap"]].round(4)

## 8. Final Interpretation

In [ ]:
# Observed vs Fitted (Model 4)
plt.figure(figsize=(6.5, 5))
sns.scatterplot(x=model4.fittedvalues, y=day["cnt"], alpha=0.7)
plt.plot(
    [day["cnt"].min(), day["cnt"].max()],
    [day["cnt"].min(), day["cnt"].max()],
    linestyle="--"
)
plt.xlabel("Fitted values")
plt.ylabel("Observed cnt")
plt.title("Observed vs Fitted Values (Model 4)")
plt.tight_layout()
plt.show()

In [ ]:
# Key coefficient table with HC3 SE
model4_hc3 = model4.get_robustcov_results(cov_type="HC3")
final_terms = [
    term for term in model4.params.index
    if any(k in term for k in ["temp", "hum", "windspeed", "workingday", "yr"])
]
discussion_table = pd.DataFrame({
    "term": model4.params.index,
    "estimate": model4.params.values,
    "SE_HC3": model4_hc3.bse,
    "p_HC3": model4_hc3.pvalues,
})
discussion_table[discussion_table["term"].isin(final_terms)].round(4)